# 1_edger_vscontrol.R

In [1]:
#######basic setting
wd="/Users/jiahuiji/Library/CloudStorage/Dropbox/lmu_project/xeno/data/Pseudobulk_for_EdgeR/cell_type_pseudobulks"
setwd(wd)

#load required packages
library(edgeR)
library(stringr)
library(limma)
library(ggplot2)

output_dic_main="/Users/jiahuiji/Library/CloudStorage/Dropbox/lmu_project/xeno/results/de_edger/"

Loading required package: limma



In [2]:
celltype_list=list.files()

In [3]:
compare_group="Extended-Standard"
compare_column_name="Hypertrophic"
genotype_group_name="Extended"
control_group_name="Standard"

## No regress co varients

In [4]:
for (j in 1:length(celltype_list))
{
celltype=celltype_list[j]
print(celltype)

#######load data
pesudo_file=celltype
pesudo_count=read.table(file=pesudo_file, header=T, sep=",")
pesudo=pesudo_count
rownames(pesudo)=pesudo[ ,1]
pesudo=pesudo[, -1]


#add meta information
clinical_info_df=read.table(file="/Users/jiahuiji/Library/CloudStorage/Dropbox/lmu_project/xeno/data/Pseudobulk_for_EdgeR/meta_info.csv", sep=",", header=T)
dif_meta=clinical_info_df
rownames(dif_meta)=dif_meta$Sample_ID
colnames(dif_meta)[which(colnames(dif_meta)==compare_column_name)]="Disease"





#######select sample
sample=colnames(pesudo)[which(colSums(pesudo)>0)]
pesudo=pesudo[,sample]
dif_meta=dif_meta[sample,]

dim(pesudo)
dim(dif_meta)




#######filter genes
mat_select=pesudo

row=c()
for (i in 1:length(rownames(mat_select)))
{
    if (sum(mat_select[i,])>0)
    {row=append(row,i)}
}
mat=mat_select[row,]
meta=dif_meta
dim(mat)
dim(dif_meta)





if (length(which(meta$Disease==control_group_name))>2 & length(which(meta$Disease==genotype_group_name))>2)
{
    #######Differential expression analysis - edgeR
    #create edgeR object
    deglist=DGEList(counts=mat, group=as.factor(meta$Disease))
    colnames(deglist)=rownames(meta)

    #filter low expression gene 
    control_idx=deglist$samples$group == control_group_name
    genotype_idx=deglist$samples$group == genotype_group_name
    # Calculate mean counts per gene per group
    mean_control=rowMeans(deglist$counts[, control_idx])
    mean_genotype=rowMeans(deglist$counts[, genotype_idx])
    # Apply filter
    keep=which(mean_control > 0.0125 | mean_genotype > 0.0125)

    deglist_keep=deglist[keep, keep.lib.sizes = FALSE]
    deglist_norm=calcNormFactors(deglist_keep, method="TMM")

    design=model.matrix(~0 + as.factor(meta$Disease))
    print(head(design))
    meta_name=colnames(design)
    cleaned_col_names=gsub(".*\\)", "", meta_name)
    colnames(design)=cleaned_col_names

    deg=estimateDisp(deglist_norm, design, robust=TRUE)
    fit=glmQLFit(deg, design)
    compare=makeContrasts(compare_group, levels=design)
    qlf=glmQLFTest(fit, contrast=compare)
    topTags(qlf)
    res=topTags(qlf, n = nrow(deglist$counts))$table
    dim(res)

    #add information of number of samples in each comparison group
    res[,paste0("Observations_", genotype_group_name)]=as.numeric(table(meta[,"Disease"])[genotype_group_name])
    res[,paste0("Observations_", control_group_name)]=as.numeric(table(meta[,"Disease"])[control_group_name])

    output_dic=paste0(output_dic_main, compare_group, "/")
    dir.create(output_dic)
    write.table(res, file=paste0(output_dic, "noregress_", celltype), sep="\t", quote=F)
}
}


[1] "pseudobulk_CM.csv"
  as.factor(meta$Disease)Control as.factor(meta$Disease)Extended
1                              1                               0
2                              1                               0
3                              1                               0
4                              1                               0
5                              0                               0
6                              0                               0
  as.factor(meta$Disease)PCMV as.factor(meta$Disease)Prematurely_terminated
1                           0                                             0
2                           0                                             0
3                           0                                             0
4                           0                                             0
5                           0                                             0
6                           0                                    

Warning message in dir.create(output_dic):
“'/Users/jiahuiji/Library/CloudStorage/Dropbox/lmu_project/xeno/results/de_edger/Extended-Standard' already exists”


[1] "pseudobulk_Endothelial_cells_baboon_.csv"
[1] "pseudobulk_Fibroblasts.csv"
  as.factor(meta$Disease)Control as.factor(meta$Disease)Extended
1                              1                               0
2                              1                               0
3                              1                               0
4                              1                               0
5                              0                               0
6                              0                               0
  as.factor(meta$Disease)PCMV as.factor(meta$Disease)Prematurely_terminated
1                           0                                             0
2                           0                                             0
3                           0                                             0
4                           0                                             0
5                           0                                             0
6        

Warning message in dir.create(output_dic):
“'/Users/jiahuiji/Library/CloudStorage/Dropbox/lmu_project/xeno/results/de_edger/Extended-Standard' already exists”


## Regress covarient

In [22]:
for (j in 1:length(celltype_list))
{
celltype=celltype_list[j]
print(celltype)

#######load data
pesudo_file=celltype
pesudo_count=read.table(file=pesudo_file, header=T, sep=",")
pesudo=pesudo_count
rownames(pesudo)=pesudo[ ,1]
pesudo=pesudo[, -1]


#add meta information
clinical_info_df=read.table(file="/Users/jiahuiji/Library/CloudStorage/Dropbox/lmu_project/xeno/data/Pseudobulk_for_EdgeR/meta_info.csv", sep=",", header=T)
dif_meta=clinical_info_df
rownames(dif_meta)=dif_meta$Sample_ID
colnames(dif_meta)[which(colnames(dif_meta)==compare_column_name)]="Disease"





#######select sample
sample=colnames(pesudo)[which(colSums(pesudo)>0)]
pesudo=pesudo[,sample]
dif_meta=dif_meta[sample,]

dim(pesudo)
dim(dif_meta)




#######filter genes
mat_select=pesudo

row=c()
for (i in 1:length(rownames(mat_select)))
{
    if (sum(mat_select[i,])>0)
    {row=append(row,i)}
}
mat=mat_select[row,]
meta=dif_meta
dim(mat)
dim(dif_meta)





if (length(which(meta$Disease==control_group_name))>2 & length(which(meta$Disease==genotype_group_name))>2)
{
    #######Differential expression analysis - edgeR
    #create edgeR object
    deglist=DGEList(counts=mat, group=as.factor(meta$Disease))
    colnames(deglist)=rownames(meta)

    #filter low expression gene 
    control_idx=deglist$samples$group == control_group_name
    genotype_idx=deglist$samples$group == genotype_group_name
    # Calculate mean counts per gene per group
    mean_control=rowMeans(deglist$counts[, control_idx])
    mean_genotype=rowMeans(deglist$counts[, genotype_idx])
    # Apply filter
    keep=which(mean_control > 0.0125 | mean_genotype > 0.0125)

    deglist_keep=deglist[keep, keep.lib.sizes = FALSE]
    deglist_norm=calcNormFactors(deglist_keep, method="TMM")

    design=model.matrix(~0 + as.factor(meta$Disease) + as.numeric(meta$Survival))
    print(head(design))
    meta_name=colnames(design)
    cleaned_col_names=gsub(".*\\)", "", meta_name)
    colnames(design)=cleaned_col_names
    colnames(design)[ncol(design)]="Survival"

    deg=estimateDisp(deglist_norm, design, robust=TRUE)
    fit=glmQLFit(deg, design)
    compare=makeContrasts(compare_group, levels=design)
    qlf=glmQLFTest(fit, contrast=compare)
    topTags(qlf)
    res=topTags(qlf, n = nrow(deglist$counts))$table
    dim(res)

    #add information of number of samples in each comparison group
    res[,paste0("Observations_", genotype_group_name)]=as.numeric(table(meta[,"Disease"])[genotype_group_name])
    res[,paste0("Observations_", control_group_name)]=as.numeric(table(meta[,"Disease"])[control_group_name])

    output_dic=paste0(output_dic_main, compare_group, "/")
    dir.create(output_dic)
    write.table(res, file=paste0(output_dic, "regress_", celltype), sep="\t", quote=F)
}
}

[1] "pseudobulk_CM.csv"
  as.factor(meta$Disease)Control as.factor(meta$Disease)Extended
1                              1                               0
2                              1                               0
3                              1                               0
4                              1                               0
5                              0                               0
6                              0                               0
  as.factor(meta$Disease)PCMV as.factor(meta$Disease)Prematurely_terminated
1                           0                                             0
2                           0                                             0
3                           0                                             0
4                           0                                             0
5                           0                                             0
6                           0                                    

Warning message in dir.create(output_dic):
“'/Users/jiahuiji/Library/CloudStorage/Dropbox/lmu_project/xeno/results/de_edger/Standard-Control' already exists”


[1] "pseudobulk_Endocardial_lymphatic_EC.csv"
  as.factor(meta$Disease)Control as.factor(meta$Disease)Extended
1                              1                               0
2                              1                               0
3                              1                               0
4                              1                               0
5                              0                               0
6                              0                               0
  as.factor(meta$Disease)PCMV as.factor(meta$Disease)Prematurely_terminated
1                           0                                             0
2                           0                                             0
3                           0                                             0
4                           0                                             0
5                           0                                             0
6                           0              

Warning message in dir.create(output_dic):
“'/Users/jiahuiji/Library/CloudStorage/Dropbox/lmu_project/xeno/results/de_edger/Standard-Control' already exists”


[1] "pseudobulk_Endothelial_cells_baboon_.csv"
[1] "pseudobulk_Fibroblasts.csv"
  as.factor(meta$Disease)Control as.factor(meta$Disease)Extended
1                              1                               0
2                              1                               0
3                              1                               0
4                              1                               0
5                              0                               0
6                              0                               0
  as.factor(meta$Disease)PCMV as.factor(meta$Disease)Prematurely_terminated
1                           0                                             0
2                           0                                             0
3                           0                                             0
4                           0                                             0
5                           0                                             0
6        

...

Warning message in dir.create(output_dic):
“'/Users/jiahuiji/Library/CloudStorage/Dropbox/lmu_project/xeno/results/de_edger/Standard-Control' already exists”


[1] "pseudobulk_Resident_Immune.csv"
  as.factor(meta$Disease)Control as.factor(meta$Disease)Extended
1                              1                               0
2                              1                               0
3                              1                               0
4                              1                               0
5                              0                               0
6                              0                               0
  as.factor(meta$Disease)PCMV as.factor(meta$Disease)Prematurely_terminated
1                           0                                             0
2                           0                                             0
3                           0                                             0
4                           0                                             0
5                           0                                             0
6                           0                       

Warning message in dir.create(output_dic):
“'/Users/jiahuiji/Library/CloudStorage/Dropbox/lmu_project/xeno/results/de_edger/Standard-Control' already exists”


[1] "pseudobulk_Vascular_EC.csv"
  as.factor(meta$Disease)Control as.factor(meta$Disease)Extended
1                              1                               0
2                              1                               0
3                              1                               0
4                              1                               0
5                              0                               0
6                              0                               0
  as.factor(meta$Disease)PCMV as.factor(meta$Disease)Prematurely_terminated
1                           0                                             0
2                           0                                             0
3                           0                                             0
4                           0                                             0
5                           0                                             0
6                           0                           

Warning message in dir.create(output_dic):
“'/Users/jiahuiji/Library/CloudStorage/Dropbox/lmu_project/xeno/results/de_edger/Standard-Control' already exists”


In [6]:
keep=which(genes_tofilter_sub[,CONTROL_COLUMN] > 0.0125 |
                          genes_tofilter_sub[,GENOTYPE_COLUMN] > 0.0125)

tt_merged[,"Observations_genotype"] <- as.numeric(table(meta.data_sub[,"Primary_Genetic_Phenotype"])[GENOTYPE])
tt_merged[,"Observations_reference"] <- as.numeric(table(meta.data_sub[,"Primary_Genetic_Phenotype"])['control'])


3. Extended vs standard
4. Rejection vs standard (extra without longrejection)
5. PCMV vs standard

6. standard vs control
7. extended vs control
8. rejection vs control
9. PCMV vs control

(with/without adjusting survival time)

ERROR: Error in parse(text = input): <text>:8:4: unexpected symbol
7: 
8: 3. Extended
      ^
